In [ ]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")

        # 1. Faster Uninstallation
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow", "huggingface_hub"],
                       capture_output=True)

        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]

        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0",
                "torchvision==0.23.0",
                "torch_xla[tpu]==2.8.0",
                "numpy", "pyarrow==16.1.0", "fsspec", # <-- Pinned pyarrow here
                "protobuf>=5.28.0",
                "datasets>=2.20.0", "transformers", "huggingface_hub>=0.28.0", "wandb", # <-- Added >=2.20.0 to datasets
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0",
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub>=0.28.0", "fsspec", "wandb", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)

        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")

        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torchvision", "torchaudio", "huggingface_hub"],
                       capture_output=True)

        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]

        try:
            print("📥 Installing GPU/CUDA Stack...")

            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])

            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub>=0.28.0",
                "wandb", "pandas<3.0.0"
            ])

            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")

        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

In [ ]:
%%writefile model.py

##################################################
# Defines HELM Phase 13A: Elastic Threshold Router
# Has a total of 32 heads, d_head = 64; only 16 will be used at a time
# True Decoupling of d_model = d_head * n_head
# Acheived via expansion layer
# Target 8 16 32
# 4 Perm heads
# No Dead Head Penalty
# Still Use Clamp
##################################################

import os
import json
import torch
import numpy as np
from safetensors.torch import load_file
import math
from math import sqrt
import random
import torch.nn.functional as F
import torch.nn as nn
try:
    from torch_xla.utils.checkpoint import checkpoint as _xla_checkpoint
except Exception:
    _xla_checkpoint = None
from transformers import AutoTokenizer
from transformers import PretrainedConfig, PreTrainedModel



# modified justnorm() function
# better than F.normalize(), max() causes micro walls during gradient descent
# better than nGPT's version, prevents division by 0 error
def justnorm(x, dim = -1, eps = 1e-12):
    res = x / (x.norm(p=2, dim=dim, keepdim=True) + eps)
    return res

# Cast the input to the correct input layer dtype
def cast_linear(x, layer):
    w = layer.weight.to(x.dtype)
    b = None if layer.bias is None else layer.bias.to(x.dtype)
    return F.linear(x,w,b)


# Hugging Face Config Class (for future deployment)
class HELMConfig(PretrainedConfig):

    model_type = "helm_7c"

    def __init__(
        self,
        # General Model Hyperparameters
        hidden_size = 1024,
        sqrt_hidden_size = 32,
        max_position_embeddings = 4096,
        initializer_range = 0.03125,
        num_hidden_layers = 12,
        num_attention_heads = 32,
        d_head = 64,
        rope_theta = 160000,
        intermediate_size = 2816,
        norm_eps = 1e-12,
        hidden_act = "swiglu",
        swiglu_s_init = 1.0,
        base_lr = 3e-4,
        min_lr = 3e-5,
        weight_decay = 0.0,
        bias = False,
        use_ckpt = False,

        # Tokenization and Data Collator Hyperparameters
        tokenizer_path = "answerdotai/ModernBERT-base",
        vocab_size = 50368,
        bos_token_id = 50281,
        eos_token_id = 50282,
        pad_token_id = 50283,
        mask_token_id = 50284,
        unk_token_id = 50285,
        mlm_probability = 0.3,
        mlm_use_span_masking = True,
        mlm_span_length = 3,

        # HELM_7c Router
        num_router_latents = 4,
        num_permanent_heads = 8,
        head_target_min = 8,
        head_target_center = 16,
        head_target_max = 32,
        easiness_cdf_breakpoints = None,
        count_loss_lambda = 0.5,
        router_grad_clip = 0.05,

        # Permanent-head training noise
        jitter_noise = 0.01,

        # nGPT self attention and FFN hyperparameters
        ngpt_sqk_init_value = 1.0,
        ngpt_sqk_init_scale = 0.03125,
        use_exclusive_attention = True,
        ngpt_alpha_value_attn = 0.05,
        ngpt_alpha_scale_attn = 0.03125,
        ngpt_alpha_value_mlp = 0.05,
        ngpt_alpha_scale_mlp = 0.03125,
        ngpt_suv_value = 1.0,
        ngpt_suv_scale = 1.0,
        ngpt_sz_init_value = 1.00,
        ngpt_sz_init_scale = 0.03125,

        dataset_total_steps = 65000,
        **kwargs
    ):
        # General model
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.d_head = d_head
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act
        self.swiglu_s_init = swiglu_s_init
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.weight_decay = weight_decay
        self.bias = bias
        self.use_ckpt = use_ckpt

        # Tokenization / MLM
        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length

        # HELM_7c Router
        self.num_router_latents = num_router_latents
        self.num_permanent_heads = num_permanent_heads
        self.head_target_min = head_target_min
        self.head_target_center = head_target_center
        self.head_target_max = head_target_max
        self.easiness_cdf_breakpoints = easiness_cdf_breakpoints
        self.count_loss_lambda = count_loss_lambda
        self.router_grad_clip = router_grad_clip
        self.jitter_noise = jitter_noise

        elastic = num_attention_heads - num_permanent_heads
        if num_permanent_heads != head_target_min:
            raise ValueError(
                "HELM_7c uses permanent heads as the structural minimum; "
                "num_permanent_heads must equal head_target_min."
            )
        if elastic <= 0:
            raise ValueError("HELM_7c requires at least one elastic head")
        if not (head_target_min <= head_target_center <= head_target_max <= num_attention_heads):
            raise ValueError("Invalid HELM_7c head targets")

        # nGPT
        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.use_exclusive_attention = use_exclusive_attention
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale
        self.dataset_total_steps = dataset_total_steps

        super().__init__(**kwargs)


class HELMEmbedding(nn.Module):

    # Initialize Embedding Layer
    def __init__(self, config):
        super().__init__()

        # Embedding Matrix size() : [vocab_size, hidden_size]
        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id
        )

    # Forward Pass (yes, its literally 3 lines)
    def forward(self, input_ids):

        # Map input_ids from Word Embeddings
        word_embeds = self.word_embeddings(input_ids)

        # Normalize (an nGPT must to allow cos. sim. to work)
        embeddings = justnorm(word_embeds)

        # Return
        return embeddings




# HELM_7c multi-latent router
class HELMMultiViewRouter(nn.Module):
    """Minimal sequence-level elastic router for HELM_7c.

    There are 8 permanent heads and 24 elastic candidates. The elastic router uses
    ordinary learned logits z_h(x). Forward routing is hard: z_h > 0. The same hard
    mask is wrapped in a sigmoid STE so CE and the count loss can train the router.

    Easiness labels are training-time supervision only. They are converted to a
    desired total head count in [8, 32]. At inference no easiness value is needed.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.scale = config.sqrt_hidden_size
        self.num_elastic_candidates = config.num_attention_heads - config.num_permanent_heads

        self.q_down_proj = nn.Linear(
            config.hidden_size,
            config.num_router_latents,
            bias=config.bias,
        )
        self.l_i_weights = nn.Parameter(torch.ones(config.num_router_latents))

        # IMPORTANT: unlike Phase 13, q_up_proj is NOT normalized. Magnitude is allowed
        # to carry information. We monitor its norms instead of pre-emptively constraining it.
        self.q_up_proj = nn.Linear(
            config.hidden_size,
            self.num_elastic_candidates,
            bias=False,
        )

        # Last-forward telemetry.
        self.save_router_logits = None
        self.save_sigmoid_scores = None
        self.save_hard_mask = None
        self.save_total_head_count = None
        self.save_target_total_head_count = None
        self.save_count_error = None
        self.save_count_loss = None

    def _easiness_to_target(self, easiness_score):
        """Map easiness label -> integer target total heads in [8, 32].

        Easiness is converted to a CDF quantile q so the target depends on relative
        difficulty rather than the raw label's dataset-specific numeric scale:
          q=0   (hardest) -> 32 total heads
          q=0.5 (median)  -> 16 total heads
          q=1   (easiest) -> 8 total heads
        """
        batch = easiness_score.numel()
        device = easiness_score.device
        e = easiness_score.to(torch.float32).reshape(batch).clamp(0.0, 1.0)

        bp = self.config.easiness_cdf_breakpoints
        if bp is not None and len(bp) >= 2:
            breaks = torch.as_tensor(bp, device=device, dtype=torch.float32)
            n_intervals = breaks.numel() - 1
            pos = torch.searchsorted(breaks, e, right=True).clamp(1, n_intervals)
            lo = breaks[pos - 1]
            hi = breaks[pos]
            frac = (e - lo) / (hi - lo + 1e-8)
            q = ((pos - 1).to(torch.float32) + frac) / float(n_intervals)
            q = q.clamp(0.0, 1.0)
        else:
            # Safe fallback if no breakpoint table was supplied.
            q = e

        h_min = float(self.config.head_target_min)
        h_ctr = float(self.config.head_target_center)
        h_max = float(self.config.head_target_max)

        hard_half = q < 0.5
        hard_target = h_ctr + (h_max - h_ctr) * ((0.5 - q) / 0.5)
        easy_target = h_ctr + (h_min - h_ctr) * ((q - 0.5) / 0.5)
        target_total = torch.where(hard_half, hard_target, easy_target)

        # Actual executed counts are integer, so make an exactly attainable target.
        return target_total.round().clamp(h_min, h_max)

    def forward(self, hidden_states, easiness_score=None):
        # ----- Existing HELM multi-latent sequence summary -----
        q_down = justnorm(self.q_down_proj.weight, dim=1).to(hidden_states.dtype)
        scanner = F.linear(hidden_states, q_down)                       # [B,S,R]
        scanner_weights = F.softmax(self.scale * scanner, dim=1)       # [B,S,R]
        latents = torch.bmm(scanner_weights.transpose(1, 2), hidden_states)  # [B,R,D]

        latent_weights = F.softmax(self.l_i_weights, dim=0)
        pooled = (latents * latent_weights.view(1, -1, 1)).sum(dim=1)  # [B,D]

        # ----- Minimal learned router -----
        router_logits = cast_linear(pooled, self.q_up_proj)                 # [B,E]
        sigmoid_scores = torch.sigmoid(router_logits)
        hard_mask = (router_logits > 0).to(router_logits.dtype)

        # Forward = exact 0/1 hard mask. Backward = sigmoid derivative.
        ste_mask = hard_mask.detach() - sigmoid_scores.detach() + sigmoid_scores

        actual_elastic_count = hard_mask.sum(dim=-1)
        actual_total_count = actual_elastic_count + float(self.config.num_permanent_heads)

        # ----- Easiness-supervised ACTUAL hard-count loss -----
        if easiness_score is not None:
            target_total_count = self._easiness_to_target(easiness_score)
            target_elastic_count = target_total_count - float(self.config.num_permanent_heads)

            # ste_mask has the hard count as its forward value but keeps a sigmoid
            # backward path. This avoids the old sum(sigmoid) soft-count loophole.
            differentiable_elastic_count = ste_mask.float().sum(dim=-1)
            count_error = differentiable_elastic_count - target_elastic_count.float()
            denom = float(self.num_elastic_candidates)
            count_loss = (
                float(self.config.count_loss_lambda)
                * (count_error / denom).square().mean()
            )
        else:
            if self.training:
                raise ValueError("HELM_7c training requires easiness_score")
            target_total_count = torch.full_like(actual_total_count, -1.0)
            count_error = torch.zeros_like(actual_total_count)
            count_loss = router_logits.new_zeros(())

        # ----- Telemetry -----
        self.save_router_logits = router_logits.detach()
        self.save_sigmoid_scores = sigmoid_scores.detach()
        self.save_hard_mask = hard_mask.detach()
        self.save_total_head_count = actual_total_count.detach()
        self.save_target_total_head_count = target_total_count.detach()
        self.save_count_error = (actual_total_count - target_total_count).detach()
        self.count_loss = count_loss
        self.save_count_loss = count_loss.detach()

        router_mask = ste_mask.view(ste_mask.size(0), -1, 1, 1)
        if self.config.num_permanent_heads > 0:
            permanent = torch.ones(
                ste_mask.size(0),
                self.config.num_permanent_heads,
                1,
                1,
                device=router_mask.device,
                dtype=router_mask.dtype,
            )
            router_mask = torch.cat((permanent, router_mask), dim=1)

        return router_mask


class RotaryEmbeddings(nn.Module):

    # Initialize the Following
    # rope_theta
    # max_position_embeddings
    # sin & cos table
    def __init__(self, dim, max_position_embeddings, rope_theta = 160000):
        super().__init__()

        # Define inverse of frequencies
        # size(): [dim/2]
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2).float() / dim))

        # Create position vector
        # size(): [max_position_embeddings]
        t = torch.arange(max_position_embeddings, dtype = inv_freq.dtype)

        freqs = torch.outer(t, inv_freq)

        freqs = torch.cat((freqs, freqs), dim = -1)


        # Save the Sine and Cosine
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    # Implement rotate_half (Allows for clean rotation mechanics)
    def rotate_half(self, x):

        # Take x as the first half
        x1 = x[..., : x.shape[-1] // 2]

        # Take y was the second half
        x2 = x[..., x.shape[-1] // 2 :]

        return torch.cat((-x2, x1), dim = -1)


    # Implement apply_rotary_embeddings
    # Does RoPE
    # Expected input size: [b, num_attention_heads, seq_len, dim]
    # Output: [b, num_attention_heads, seq_len, dim]
    def forward(self, x):

        # Get token length
        seq_len = x.shape[-2]

        # Take a slice of the cos and sin tables
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)

        # Return RoPE matrix
        return (x * x_cos) + (self.rotate_half(x) * x_sin)



# Self Attention
# Literally Just Self Attention
# QKV cross self attention
# Use RoPE
# Output Matrix
# Speicfics about training (masked training)
# MODIFICATION: USE FLEX ATTENTION TO ALLOW FOR BATCHED INFERENCE
class HELMSelfAttention(nn.Module):

    # Initialize the following:
    #   - QKV matrix
    #   - Output matrix
    #   - Scaling vector sqk for q and k
    #   - RoPE Module
    def __init__(self, config):
        super().__init__()

        # Grabbing config values from convience
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_permanent_heads = config.num_permanent_heads
        self.d_head = config.d_head if config.d_head is not None else (config.hidden_size // config.num_attention_heads)
        self.total_head_dim = self.num_attention_heads * self.d_head   
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale
        self.config = config

        self._eval_backend = "dense"
        self._flex_compiled = False
        self._flex_fn = None
        self._block_mask_fn = None


        # QKV Matrix
        self.qkv = nn.Linear(
            config.hidden_size,
            self.total_head_dim * 3,
            bias = config.bias
        )

        # RoPE Module
        self.RoPE = RotaryEmbeddings(
            self.d_head,
            config.max_position_embeddings,
            config.rope_theta
        )

        # SQK scalers right after RoPE
        self.sqk = nn.Parameter(self.ngpt_sqk_init_scale*torch.ones(self.total_head_dim))  # was: self.hidden_size

        # Output Matrix
        self.output = nn.Linear(
            self.total_head_dim,      # was: config.hidden_size
            config.hidden_size,
            bias = config.bias
        )

    # Configure the eval-time attention backend. Call via model.enable_efficient_inference(...).
    #   backend="flex"  : FlexAttention; set compile=True on GPU for the fused kernel (recommended).
    #   backend="gather": compact gather/scatter SDPA, no torch.compile needed.
    #   backend="dense" : compute-all-then-mask (default; what training uses).
    def set_eval_backend(self, backend="flex", compile=True):
        compile = bool(compile)
        # Only drop the cached torch.compile()'d function/block-mask builder when the
        # backend or compile flag actually changes -- resetting on every call (even when
        # nothing changed) forces a full recompilation on the very next forward pass,
        # which is silently expensive if this is called before every timed benchmark run.
        changed = (backend != getattr(self, "_eval_backend", None)
                   or compile != getattr(self, "_flex_compiled", None))
        self._eval_backend = backend
        self._flex_compiled = compile
        if changed:
            self._flex_fn = None
            self._block_mask_fn = None

    def _flex_attn(self, q, k, v, block_mask, scale):
        if self._flex_fn is None:
            from torch.nn.attention.flex_attention import flex_attention
            self._flex_fn = torch.compile(flex_attention) if self._flex_compiled else flex_attention
        return self._flex_fn(q, k, v, block_mask=block_mask, scale=scale)

    def _build_block_mask(self, mask_mod, B, H, S, device):
        if self._block_mask_fn is None:
            from torch.nn.attention.flex_attention import create_block_mask
            # compiling create_block_mask avoids materializing the full SxS mask for long sequences
            self._block_mask_fn = torch.compile(create_block_mask) if self._flex_compiled else create_block_mask
        return self._block_mask_fn(mask_mod, B, H, S, S, device=device)

    # Define Training
    def forward(self, hidden_states, attention_mask, router_mask):

        # Obtain projection from hidden_states onto QKV
        # size(): [b, seq_len, hidden_size * 3]
        qkv_proj = cast_linear(hidden_states, self.qkv)

        # Obtain Hidden Size
        batch_size, seq_len, _ = hidden_states.size()

        # Split Projects
        # q, k, v size(): [b, seq_len, hidden_size]
        q, k, v = qkv_proj.split(self.total_head_dim, dim=-1)

        # Define sqk for scaling q, k, and v
        # size(): [hidden_size]
        sqk = (self.sqk * (self.ngpt_sqk_init_value/self.ngpt_sqk_init_scale))
        # Resizing is required for when we element-wise multiply this by q and k matrice:s [1, num_attention_heads, 1, d_head] * [b, num_attention_heads, seq_len, hidden_size]
        # size(): [hidden_size]-> [1, num_attention_heads, 1, d_head]
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)


        eval_backend = self._eval_backend

        # Reshape q,k,v
        # q, k, v size(): [b, seq_len, num_attention_heads, d_head]
        q = q.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        k = k.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        v = v.view(batch_size, seq_len, self.num_attention_heads, self.d_head)

        # Reshape q,k,v
        # q, k, v size(): [b, num_attention_heads, seq_len, d_head]
        q = q.permute(0,2,1,3)
        k = k.permute(0,2,1,3)
        v = v.permute(0,2,1,3)


        # TRAINING / TPU MODE
        if (self.training or eval_backend == "dense"):

            # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Apply Attention
            # Scale by sqrt(dk)
            # A whole lot happens here. final size(): [b, num_attention_heads, seq_len, d_head]
            context_layer = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head),
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            if router_mask is not None:
                # Apply Broadcasting Mask (expand_as() good for XLA)
                # size(): [b, num_attention_heads, seq_len, d_head]
                context_layer = context_layer * router_mask.expand_as(context_layer)

            # Apply Jitter Noise to the Permanent heads during training
            if self.training and self.num_permanent_heads > 0:

                # Take the permanent heads:
                permanent_heads = context_layer[:,:self.num_permanent_heads, :, :]

                # Take the elastic heads:
                elastic_heads = context_layer[:, self.num_permanent_heads:, :, :]

                # Apply dropout
                permanent_heads = F.dropout(permanent_heads, p = self.config.jitter_noise, training = self.training)

                # Combine back together
                context_layer = torch.cat((permanent_heads, elastic_heads),dim = 1)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # FLEX ATTENTION (for GPUs)
        elif eval_backend == "flex" and batch_size > 1:

             # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Router_mask scores, 1 or 0 or sigmoid scaling
            # [b, num_attention_heads, 1 , 1] -> [batch, num_attention_heads]
            active = (router_mask[:, :, 0, 0] > 0)

            # Boolean attention mask
            # [batch_size, 1, 1, seq_len] -> [batch, seq_len]
            key_valid = (attention_mask[:, 0, 0, :] >=0)

            # mask_mod: attend / calculate only if the head is on and its not a padding token
            def mask_mod(bi, hi, qi, ki):
                return active[bi, hi] & key_valid[bi, ki]

            # Prep the block to be passed into flex attention
            block_mask = self._build_block_mask(
                mask_mod, batch_size, self.num_attention_heads,seq_len, q.device
            )

            # Apply flex attention
            context_layer = self._flex_attn(
                q, k, v, block_mask = block_mask, scale = math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            # Apply router mask to 0 the heads of the context layer
            # [batch, num attention heads, seq_len, head dim] (router_mask [batch, num_attention_heads, 1,1] was broadcasted)
            context_layer = context_layer * router_mask.expand_as(context_layer)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # Single query effieincy
        else:

            # This path only looks at batch element 0's router decisions (see below), so
            # it is only correct for batch_size == 1 -- each example's active heads are
            # data-dependent, so silently reusing example 0's mask for other examples
            # would produce wrong outputs for them instead of a loud failure.
            assert batch_size == 1, (
                f"HELMSelfAttention's 'gather' eval backend only supports batch_size == 1 "
                f"(got batch_size={batch_size}); use backend='flex' for batched inference."
            )

            # Find the heads that are on
            # nonzero(): [1, num_attention_heads, 1, 1] -> [num_active_heads, 1]
            # squeeze(): [num_active_heads, 1] -> [num_active_heads] (indices)
            active_indices = torch.nonzero(router_mask[0, :, 0, 0]).squeeze(-1)

            # q, k, v are already [b, num_attention_heads, seq_len, d_head] from the
            # shared reshape/permute above -- no need to reshape them again here.

            # 2. Extract the parts used by the active heads
            # size(): [1, num_attention_heads, seq_len, d_head] ->  [1, num_active_heads, seq_len, d_head]
            q_sliced = q[:, active_indices, :, :]
            k_sliced = k[:, active_indices, :, :]
            v_sliced = v[:, active_indices, :, :]

            # Normalize q and k
            q_sliced = justnorm(q_sliced)
            k_sliced = justnorm(k_sliced)

            # Apply RoPE
            q_sliced = self.RoPE(q_sliced)
            k_sliced = self.RoPE(k_sliced)

            # Apply sqk scaling factor to q and k
            sqk_sliced = sqk[:, active_indices, :, :]
            q_sliced = sqk_sliced.to(q_sliced.dtype) * q_sliced
            k_sliced = sqk_sliced.to(k_sliced.dtype) * k_sliced

            # Flash Attention (only for GPUs where on-the-fly splicing can exist)
            # size(): [b, num_active_heads, seq_len, d_head]
            context_sliced = F.scaled_dot_product_attention(
                q_sliced, k_sliced, v_sliced,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v_sliced, dim=-1)
                context_sliced = context_sliced - (context_sliced * Vn).sum(dim=-1, keepdim=True) * Vn

            # STE tie to the router
            # Note: If use_sigmoid_scaling = True: Scales the router mask back to the sigmoid values
            # (since active indices were just indices of the values, not the real values)
            # If use_sigmooid_scaling = False, then multiplying by 1 does mathimatically nothing
            active_weights = router_mask[:, active_indices, :, :]
            context_sliced = context_sliced * active_weights

            # 5. Reshape for the output linear layer
            # [1, num_active, seq_len, d_head] -> [1, seq_len, num_active, d_head]
            context_reshaped = context_sliced.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions: [1, seq_len, num_active * d_head]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # 6. Map the active head indices to their exact hidden dimension indices
            # Example: Head 1 with d_head=64 generates indices 64 through 127
            dim_offsets = torch.arange(self.d_head, device=hidden_states.device)
            active_dims = (active_indices.unsqueeze(1) * self.d_head + dim_offsets).view(-1)

            # 7. Slice the input columns of the output weight matrix
            # original shape [hidden_size, hidden_size] -> [hidden_size, num_active * d_head]
            sliced_weight = self.output.weight[:, active_dims].to(context_reshaped.dtype)
            sliced_bias = None if self.output.bias is None else self.output.bias.to(context_reshaped.dtype)

            # 8. Perform the compressed functional linear projection
            context_layer = F.linear(context_reshaped, sliced_weight, bias=sliced_bias)

        # Return context_layer (normalization occurs in HELMMLP)
        return context_layer



# HELMMLP (FFN of nGPT architecture)
# All of this stays the same from the original nGPT paper
class HELMMLP(nn.Module):

    # Define the Following:
    #   - Constants from config (for convience?)
    #       * hidden_size
    #       * ngpt_alpha_value_attn
    #       * ngpt_alpha_scale_attn
    #       * ngpt_alpha_value_mlp
    #       * ngpt_alpha_scale_mlp
    #       * ngpt_suv_value
    #       * ngpt_suv_scale
    #   - Eigen learning rate after attention (attn_alpha)
    #   - Eigen learning rate after mlp (mlp_alpha)
    #   - MLP expansion layer (mlp_exp)
    #   - suv scaling vectors for SwiGLU (suv)
    #   - SiLU() activation (silu)
    #   - MLP projection layer (mlp_expand)
    def __init__(self, config):
        super().__init__()

        # Gather Config Values for convience
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        # Alpha Eigen Update after Attention (1st Optimizer Step)
        self.attn_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_attn*torch.ones(self.hidden_size))

        # Alpha Eigen Update after MLP (2nd Optimizer Step)
        self.mlp_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_mlp*torch.ones(self.hidden_size))

        # MLP expansion layer
        self.mlp_exp = nn.Linear(
            self.hidden_size,
            2 * self.intermediate_size,
            bias = config.bias
        )

        # suv scaling vectors during SwiGLU
        self.suv = torch.nn.Parameter(self.ngpt_suv_scale*torch.ones(2 * self.intermediate_size))

        # Define SiLU()
        self.silu = nn.SiLU()

        # MLP projection layer (shrink)
        self.mlp_proj  = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias
        )

    # Peform MLP from the output of the output matrix to the end of the transformer block
    def forward(self, hidden_states, hidden_states_attention):

        # Even more convience
        hidden_size = self.hidden_size
        ngpt_alpha_value_attn = self.ngpt_alpha_value_attn
        ngpt_alpha_scale_attn = self.ngpt_alpha_scale_attn
        ngpt_alpha_value_mlp = self.ngpt_alpha_value_mlp
        ngpt_alpha_scale_mlp = self.ngpt_alpha_scale_mlp
        ngpt_suv_value = self.ngpt_suv_value
        ngpt_suv_scale = self.ngpt_suv_scale

        # Mostly Lifted from the nGPT model.py

        # Apply Normalization to hidden states before and after attention
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states)
        B_norm = justnorm(hidden_states_attention)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.attn_alpha * (ngpt_alpha_value_attn / ngpt_alpha_scale_attn)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_a * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt1 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt1 = justnorm(hidden_states_opt1)

        # Get u and v matrices by multiplying by mlp_exp
        # size(): [b, seq_len, hidden_size] * [hidden_size, 2 * intermediate_size] = [b, seq_len, 2 * intermediate_size]
        uv_pre = cast_linear(hidden_states_opt1 ,self.mlp_exp)
        # prepare scaling vector suv
        # size(): [intermediate_size * 2] (remember, they are concatenated)
        suv = self.suv * (ngpt_suv_value/ngpt_suv_scale) * (hidden_size ** 0.5)
        # We need to keep suv to be bf16. The line above promoted suc fp32 and the autocaster didn't fix it
        suv = suv.to(uv_pre.dtype)

        # element-wise uv by scaling vector suv
        # size(): [b, seq_len, 2 * intermediate_size]
        uv_post_suv = suv * uv_pre

        # Chunk uv into u and v
        # both size(): [b, seq_len, intermediate_size]
        u, v = torch.chunk(uv_post_suv, 2, dim=-1)

        # Apply u * silu(v), the whole point of SwiGLU (element-wise)
        # size(): [b, seq_len, intermediate_size]
        x_mlp = u * self.silu(v)

        # Project x_mlp to the mlp_proj layer (shrink)
        # size(): [b, seq_len, intermediate_size] * [intermediate_size, hidden_size] = [b, seq_len, hidden_size]
        h_mlp = cast_linear(x_mlp, self.mlp_proj)

        # Apply Normalization to hidden states after attention and after mlp
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states_opt1)
        B_norm = justnorm(h_mlp)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.mlp_alpha * (ngpt_alpha_value_mlp / ngpt_alpha_scale_mlp)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_m * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt2 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt2 = justnorm(hidden_states_opt2)

        # Return new hidden_state
        return hidden_states_opt2



# HELMBLOCK = HELMMultiViewRouter + HELMSelfAttention + HELMMLP
class HELMBlock(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.mlt_vw_rtr = HELMMultiViewRouter(config)
        self.attn = HELMSelfAttention(config)
        self.mlp = HELMMLP(config)

    def forward(self, hidden_states, attention_mask, easiness_score):
        router_mask = self.mlt_vw_rtr(hidden_states, easiness_score)
        count_loss = self.mlt_vw_rtr.count_loss
        attn_output = self.attn(hidden_states, attention_mask, router_mask)
        layer_output = self.mlp(hidden_states, attn_output)
        return layer_output, count_loss


class HELMModel(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.use_ckpt = config.use_ckpt
        self.embedding = HELMEmbedding(config)
        self.blocks = nn.ModuleList([HELMBlock(config) for _ in range(config.num_hidden_layers)])

    def forward(self, input_ids, attention_mask, easiness_score=None):
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).to(torch.bfloat16)
        attention_mask = attention_mask.masked_fill(attention_mask == 0, float('-inf'))
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        hidden_states = self.embedding(input_ids).to(torch.bfloat16)
        total_count_loss = hidden_states.new_zeros(())

        for block in self.blocks:
            if self.use_ckpt and self.training:
                _ckpt = (_xla_checkpoint if (_xla_checkpoint is not None
                         and hidden_states.device.type == "xla")
                         else torch.utils.checkpoint.checkpoint)
                hidden_states, count_loss = _ckpt(
                    block,
                    hidden_states,
                    attention_mask,
                    easiness_score,
                    use_reentrant=True if hidden_states.device.type == "xla" else False,
                )
            else:
                hidden_states, count_loss = block(hidden_states, attention_mask, easiness_score)
            total_count_loss = total_count_loss + count_loss

        # Count supervision is per-layer; average so lambda is independent of depth.
        total_count_loss = total_count_loss / float(len(self.blocks))
        return hidden_states, total_count_loss


class HELMForMaskedLM(PreTrainedModel):

    config_class = HELMConfig

    def __init__(self, config):
        super().__init__(config)
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale
        self.model = HELMModel(config)
        self.classifier = nn.Linear(config.hidden_size, config.vocab_size, bias=config.bias)
        self.sz = nn.Parameter(torch.ones(config.vocab_size))
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)

    def enable_efficient_inference(self, backend="flex", compile=True):
        for block in self.model.blocks:
            block.attn.set_eval_backend(backend=backend, compile=compile)
        return self

    @torch.no_grad()
    def normalize_ngpt_matrices(self):
        # q_up_proj is intentionally EXCLUDED: HELM_7c allows router magnitude.
        keys_to_normalize = (
            "word_embeddings.weight",
            "classifier.weight",
            "attn.qkv.weight",
            "attn.output.weight",
            "mlp.mlp_exp.weight",
            "mlp.mlp_proj.weight",
            "mlt_vw_rtr.q_down_proj.weight",
        )
        for name, param in self.named_parameters():
            if name.endswith(keys_to_normalize):
                param.copy_(justnorm(param, dim=1, eps=1e-12))

    @torch.no_grad()
    def get_telemetry(self):
        telemetry = {}

        for i, block in enumerate(self.model.blocks):
            router = block.mlt_vw_rtr
            logits = router.save_router_logits.float().cpu()
            sigmoid = router.save_sigmoid_scores.float().cpu()
            hard = router.save_hard_mask.float().cpu()
            actual = router.save_total_head_count.float().cpu()
            target = router.save_target_total_head_count.float().cpu()
            error = router.save_count_error.float().cpu()
            q_up_norms = router.q_up_proj.weight.detach().float().norm(dim=1).cpu()

            telemetry[f"layer_{i}_router_logits"] = logits
            telemetry[f"layer_{i}_sigmoid_scores"] = sigmoid
            telemetry[f"layer_{i}_hard_mask"] = hard
            telemetry[f"layer_{i}_elastic_head_ratio"] = hard.mean().item()
            telemetry[f"layer_{i}_total_head_count_mean"] = actual.mean().item()
            telemetry[f"layer_{i}_target_head_count_mean"] = target.mean().item()
            telemetry[f"layer_{i}_count_error_mean"] = error.mean().item()
            telemetry[f"layer_{i}_count_error_mae"] = error.abs().mean().item()
            telemetry[f"layer_{i}_count_loss"] = router.save_count_loss.float().item()
            telemetry[f"layer_{i}_router_weight_norms"] = q_up_norms
            telemetry[f"layer_{i}_router_weight_norm_mean"] = q_up_norms.mean().item()
            telemetry[f"layer_{i}_router_weight_norm_std"] = q_up_norms.std(unbiased=False).item()
            telemetry[f"layer_{i}_l_i_weights"] = router.l_i_weights.detach().float().cpu()

        return telemetry

    def forward(self, input_ids, attention_mask, current_step=None, easiness_score=None):
        # current_step is accepted only for backward compatibility with older callers.
        features, total_count_loss = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            easiness_score=easiness_score,
        )

        sz = self.sz * (self.ngpt_sz_init_value / self.ngpt_sz_init_scale)
        unscaled_logits = cast_linear(features, self.classifier)
        logits = sz.to(unscaled_logits.dtype) * unscaled_logits
        return logits, total_count_loss



In [ ]:
%%writefile analyze_helm7c_heads.py
#!/usr/bin/env python3
"""
HELM_7c head-bank assumption audit.

This is the post-dense16 version of the HELM_7c diagnostic.  The main goal is
NOT to search for another router mechanism.  It asks whether the conclusions we
previously drew from raw effective rank were actually justified.

Core questions
--------------
1. Is HELM_7c genuinely directionally low-rank, or does it merely have unequal
   head magnitudes (the same pattern seen in the healthy dense-16 baseline)?
2. Does any rank reduction already exist in the pre-W_O attention contexts, or
   is it introduced mainly when heads write back through their W_O blocks?
3. Are the 24 elastic candidates themselves diverse when ALL are evaluated,
   even if the router only executes a subset of them?
4. How different is the geometry of the full POTENTIAL head bank from the
   geometry actually EXECUTED by the router?
5. Do activation frequency and head strength remain correlated?
6. Does HELM still select useful identities (routed vs random-same-count), and
   is forcing all 32 heads on actually helpful at this checkpoint?

Rank terminology
----------------
For a group of head vectors Y_h:

RAW effective rank
    Uses the ordinary Gram matrix YY^T.  It falls when heads point in similar
    directions OR when a few heads carry much more energy than the others.

DIRECTIONAL effective rank
    First removes each head's magnitude (equivalently uses the cosine Gram
    matrix), then computes effective rank.  This asks whether the heads point in
    genuinely different functional directions without penalizing unequal RMS.

POTENTIAL geometry
    Recomputes every attention head before the router mask.  This asks what the
    trained head bank is capable of producing for the same hidden states.

EXECUTED geometry
    Applies the actual HELM hard mask before measuring the head vectors.  This
    asks what dimensionality/diversity the routed forward computation actually
    exposes across the sampled examples.

The script deliberately uses a deterministic diagnostic MLM mask.  Its CE is
for relative comparisons inside this script, not for reproducing the trainer's
exact validation loss.

Expected setup
--------------
- HELM_7c model.py is in the same working directory as this script.
- Default checkpoint: JamesResearch1216/HELM_7c/checkpoint-006500.pt
- Default validation shard:
  JamesResearch1216/HELM-Easiness-Data-10B-Labeled-v6/
  data/seq_1024/validation-00000.parquet

Typical Kaggle TPU run
----------------------
python analyze_helm7c_heads.py \
    --device xla \
    --batch-size 2 \
    --num-examples 16 \
    --functional-batches 4

Heavier optional exact ablation
-------------------------------
python analyze_helm7c_heads.py \
    --device xla \
    --batch-size 2 \
    --num-examples 16 \
    --functional-batches 4 \
    --exact-ablation \
    --ablation-layers 0,5,11 \
    --ablation-batches 1
"""

from __future__ import annotations

import argparse
import csv
import json
import math
import os
import random
import re
import shutil
import sys
from contextlib import contextmanager
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

try:
    import pyarrow.parquet as pq
except Exception as exc:
    raise RuntimeError("pyarrow is required: pip install pyarrow") from exc

try:
    from huggingface_hub import hf_hub_download
except Exception as exc:
    raise RuntimeError("huggingface_hub is required: pip install huggingface_hub") from exc


# model.py should be created by the HELM_7c architecture cell in the same cwd.
SCRIPT_DIR = Path(__file__).resolve().parent
CWD = Path.cwd()
for p in (CWD, SCRIPT_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

try:
    from model import HELMConfig, HELMForMaskedLM, justnorm, cast_linear
except Exception as exc:
    raise RuntimeError(
        "Could not import HELM_7c model.py. Run/write the HELM_7c architecture "
        "cell first so model.py is available in the working directory."
    ) from exc


# -----------------------------------------------------------------------------
# Defaults
# -----------------------------------------------------------------------------
MODEL_REPO = "JamesResearch1216/HELM_7c"
CHECKPOINT_FILE = "checkpoint-006500.pt"
TRAINING_STATE_FILE = "training_state.json"
DATA_REPO = "JamesResearch1216/HELM-Easiness-Data-10B-Labeled-v6"
VALIDATION_FILE = "data/seq_1024/validation-00000.parquet"


# -----------------------------------------------------------------------------
# Generic helpers
# -----------------------------------------------------------------------------
def get_hf_token() -> Optional[str]:
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def strip_state_prefixes(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    out = {}
    for key, value in state.items():
        k = key
        changed = True
        while changed:
            changed = False
            for prefix in ("module.", "_orig_mod."):
                if k.startswith(prefix):
                    k = k[len(prefix):]
                    changed = True
        out[k] = value
    return out


def checkpoint_step_from_name(name: str) -> int:
    m = re.search(r"checkpoint-(\d+)\.pt$", name)
    return int(m.group(1)) if m else 6500


def rankdata_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x), dtype=np.float64)
    sorted_x = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sorted_x[j] == sorted_x[i]:
            j += 1
        ranks[order[i:j]] = 0.5 * (i + j - 1)
        i = j
    return ranks


def spearman_np(x, y) -> float:
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    good = np.isfinite(x) & np.isfinite(y)
    x, y = x[good], y[good]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return float("nan")
    return float(np.corrcoef(rankdata_np(x), rankdata_np(y))[0, 1])


def effective_rank_from_gram(gram: np.ndarray) -> Tuple[float, float, np.ndarray]:
    """Entropy effective rank, participation rank, eigenvalues."""
    g = 0.5 * (gram + gram.T)
    vals = np.linalg.eigvalsh(g)
    vals = np.clip(vals, 0.0, None)
    total = vals.sum()
    if total <= 1e-12:
        return 0.0, 0.0, vals
    p = vals / total
    p_pos = p[p > 1e-12]
    entropy_rank = float(np.exp(-(p_pos * np.log(p_pos)).sum()))
    participation_rank = float((total * total) / (np.square(vals).sum() + 1e-12))
    return entropy_rank, participation_rank, vals


def cosine_from_gram(gram: np.ndarray) -> np.ndarray:
    diag = np.clip(np.diag(gram), 0.0, None)
    denom = np.sqrt(np.outer(diag, diag)) + 1e-12
    cos = gram / denom
    cos = np.clip(cos, -1.0, 1.0)
    return 0.5 * (cos + cos.T)


def offdiag_stats(cos: np.ndarray) -> Dict[str, float]:
    h = cos.shape[0]
    if h <= 1:
        return {
            "mean_abs": 0.0,
            "median_abs": 0.0,
            "max_abs": 0.0,
            "frac_abs_gt_0.5": 0.0,
            "frac_abs_gt_0.8": 0.0,
        }
    vals = cos[~np.eye(h, dtype=bool)]
    av = np.abs(vals)
    return {
        "mean_abs": float(av.mean()) if av.size else 0.0,
        "median_abs": float(np.median(av)) if av.size else 0.0,
        "max_abs": float(av.max()) if av.size else 0.0,
        "frac_abs_gt_0.5": float((av > 0.5).mean()) if av.size else 0.0,
        "frac_abs_gt_0.8": float((av > 0.8).mean()) if av.size else 0.0,
    }


def components_for_fraction(eigvals: np.ndarray, fraction: float) -> int:
    vals = np.sort(np.clip(eigvals, 0.0, None))[::-1]
    total = vals.sum()
    if total <= 1e-12:
        return 0
    c = np.cumsum(vals) / total
    return int(np.searchsorted(c, fraction, side="left") + 1)


def gini_nonnegative(values: np.ndarray) -> float:
    x = np.asarray(values, dtype=np.float64).reshape(-1)
    x = np.clip(x, 0.0, None)
    if x.size == 0 or x.sum() <= 1e-12:
        return 0.0
    x = np.sort(x)
    n = x.size
    idx = np.arange(1, n + 1, dtype=np.float64)
    return float((2.0 * np.sum(idx * x) / (n * x.sum())) - (n + 1.0) / n)


def double_argsort_rank(x: torch.Tensor, descending: bool = True) -> torch.Tensor:
    order = torch.argsort(x, dim=-1, descending=descending)
    return torch.argsort(order, dim=-1)


def parse_layers(spec: str, n_layers: int) -> List[int]:
    if spec.lower() == "all":
        return list(range(n_layers))
    out = []
    for part in spec.split(","):
        part = part.strip()
        if not part:
            continue
        li = int(part)
        if not 0 <= li < n_layers:
            raise ValueError(f"Layer {li} out of range [0,{n_layers - 1}]")
        out.append(li)
    return sorted(set(out))


def json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    return obj


# -----------------------------------------------------------------------------
# Plot helpers
# -----------------------------------------------------------------------------
def save_heatmap(path: Path, matrix: np.ndarray, title: str, vmin=None, vmax=None) -> None:
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(matrix, aspect="auto", interpolation="nearest", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("Head")
    ax.set_ylabel("Head")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def save_bar(path: Path, values: np.ndarray, title: str, ylabel: str) -> None:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(np.arange(len(values)), values)
    ax.set_title(title)
    ax.set_xlabel("Head")
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def save_rank_depth_plot(path: Path, rows: List[dict], metric: str, title: str) -> None:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for group in ("all", "permanent", "elastic"):
        sub = [r for r in rows if r["group"] == group and r["geometry"] == "potential"]
        if not sub:
            continue
        sub = sorted(sub, key=lambda r: r["layer"])
        ax.plot([r["layer"] for r in sub], [r[metric] for r in sub], marker="o", label=group)
    ax.set_xlabel("Layer")
    ax.set_ylabel("Effective rank")
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


# -----------------------------------------------------------------------------
# Device
# -----------------------------------------------------------------------------
@dataclass
class DeviceContext:
    device: torch.device
    kind: str
    xm: object = None
    autocast_dtype: Optional[torch.dtype] = None

    def autocast(self):
        if self.kind == "cuda":
            return torch.autocast(device_type="cuda", dtype=self.autocast_dtype)
        # XLA bf16 is already handled by the model tensors; avoid CUDA-style autocast.
        from contextlib import nullcontext
        return nullcontext()

    def mark_step(self):
        if self.kind == "xla" and self.xm is not None:
            self.xm.mark_step()


def resolve_device(requested: str) -> DeviceContext:
    if requested == "auto":
        # Prefer XLA when the runtime exposes a TPU.
        try:
            import torch_xla.core.xla_model as xm
            dev = xm.xla_device()
            return DeviceContext(dev, "xla", xm=xm, autocast_dtype=torch.bfloat16)
        except Exception:
            pass
        if torch.cuda.is_available():
            return DeviceContext(torch.device("cuda"), "cuda", autocast_dtype=torch.bfloat16)
        return DeviceContext(torch.device("cpu"), "cpu", autocast_dtype=None)

    if requested == "xla":
        try:
            import torch_xla.core.xla_model as xm
        except Exception as exc:
            raise RuntimeError(
                "--device xla requested but torch_xla could not be imported. "
                "Run your normal Kaggle TPU repair/restart cell first."
            ) from exc
        return DeviceContext(xm.xla_device(), "xla", xm=xm, autocast_dtype=torch.bfloat16)

    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA requested but torch.cuda.is_available() is False")
        return DeviceContext(torch.device("cuda"), "cuda", autocast_dtype=torch.bfloat16)

    if requested == "cpu":
        return DeviceContext(torch.device("cpu"), "cpu", autocast_dtype=None)

    raise ValueError(f"Unknown device: {requested}")


# -----------------------------------------------------------------------------
# Download / model / data
# -----------------------------------------------------------------------------
def download_assets(
    cache_dir: Path,
    token: Optional[str],
    model_repo: str,
    checkpoint_file: str,
    data_repo: str,
    validation_file: str,
):
    cache_dir.mkdir(parents=True, exist_ok=True)

    print(f"Downloading checkpoint: {model_repo}/{checkpoint_file}")
    ckpt_path = hf_hub_download(
        repo_id=model_repo,
        filename=checkpoint_file,
        repo_type="model",
        token=token,
        local_dir=str(cache_dir / "model_repo"),
    )

    training_state_path = None
    try:
        training_state_path = hf_hub_download(
            repo_id=model_repo,
            filename=TRAINING_STATE_FILE,
            repo_type="model",
            token=token,
            local_dir=str(cache_dir / "model_repo"),
        )
    except Exception as exc:
        print(f"WARNING: could not download training_state.json: {exc}")

    print(f"Downloading validation shard: {data_repo}/{validation_file}")
    validation_path = hf_hub_download(
        repo_id=data_repo,
        filename=validation_file,
        repo_type="dataset",
        token=token,
        local_dir=str(cache_dir / "dataset"),
    )

    return (
        Path(ckpt_path),
        Path(training_state_path) if training_state_path else None,
        Path(validation_path),
    )


def load_breakpoints(training_state_path: Optional[Path]) -> Optional[List[float]]:
    if training_state_path is None or not training_state_path.exists():
        return None
    try:
        with training_state_path.open("r") as f:
            state = json.load(f)
        easiness_dict = state.get("easiness_dict")
        if isinstance(easiness_dict, dict):
            bp = easiness_dict.get("breakpoints")
            if bp:
                print(f"Loaded {len(bp)} easiness CDF breakpoints from training_state.json")
                return bp
    except Exception as exc:
        print(f"WARNING: failed to parse easiness breakpoints: {exc}")
    return None


def load_model(
    ckpt_path: Path,
    breakpoints: Optional[List[float]],
    dev: DeviceContext,
):
    config = HELMConfig(easiness_cdf_breakpoints=breakpoints)
    model = HELMForMaskedLM(config)

    print(f"Loading checkpoint from {ckpt_path}")
    payload = torch.load(str(ckpt_path), map_location="cpu")
    if not isinstance(payload, dict):
        raise RuntimeError("Checkpoint is not a dict")

    state = strip_state_prefixes(payload.get("model_state", payload))
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print(f"State load: {len(missing)} missing, {len(unexpected)} unexpected")
        if missing:
            print("  Missing (first 10):", missing[:10])
        if unexpected:
            print("  Unexpected (first 10):", unexpected[:10])
        if len(missing) > 5 or len(unexpected) > 5:
            raise RuntimeError("Large state-dict mismatch; make sure model.py is HELM_7c")

    del payload
    model.to(dev.device)
    model.eval()
    # Fixed-shape compute-all-then-mask path: appropriate for TPU analysis and for
    # measuring the pre-router head bank.
    model.enable_efficient_inference("dense", compile=False)
    return model, config


def deterministic_span_mask(
    ids: torch.Tensor,
    config: HELMConfig,
    seed: int,
    probability: float = 0.30,
    span_length: int = 3,
) -> Tuple[torch.Tensor, torch.Tensor]:
    ids = ids.clone().long()
    labels = torch.full_like(ids, -100)
    g = torch.Generator(device="cpu")
    g.manual_seed(int(seed))

    special = {
        int(config.bos_token_id),
        int(config.eos_token_id),
        int(config.pad_token_id),
        int(config.mask_token_id),
        int(config.unk_token_id),
    }
    candidate = [i for i, tok in enumerate(ids.tolist()) if int(tok) not in special]
    if not candidate:
        return ids, labels

    target = max(1, int(round(probability * len(candidate))))
    candidate_set = set(candidate)
    perm = torch.randperm(len(candidate), generator=g).tolist()
    chosen = set()
    for pi in perm:
        if len(chosen) >= target:
            break
        start = candidate[pi]
        for pos in range(start, min(start + span_length, ids.numel())):
            if pos in candidate_set:
                chosen.add(pos)
                if len(chosen) >= target:
                    break

    chosen = sorted(chosen) or [candidate[0]]
    pos = torch.tensor(chosen, dtype=torch.long)
    original = ids[pos].clone()
    labels[pos] = original

    r = torch.rand(len(pos), generator=g)
    mask_sel = r < 0.80
    random_sel = (r >= 0.80) & (r < 0.90)
    ids[pos[mask_sel]] = int(config.mask_token_id)
    if random_sel.any():
        random_tokens = torch.randint(
            low=0,
            high=int(config.vocab_size),
            size=(int(random_sel.sum()),),
            generator=g,
        )
        ids[pos[random_sel]] = random_tokens
    return ids, labels


def prepare_batches(
    validation_path: Path,
    config: HELMConfig,
    num_examples: int,
    batch_size: int,
    seq_len: int,
    seed: int,
) -> List[Dict[str, torch.Tensor]]:
    table = pq.read_table(str(validation_path), columns=["input_ids", "easiness_score"])
    if "input_ids" not in table.column_names or "easiness_score" not in table.column_names:
        raise RuntimeError(
            f"Validation parquet columns are {table.column_names}; "
            "expected input_ids and easiness_score"
        )

    total_rows = table.num_rows
    n = min(num_examples, total_rows)
    rng = np.random.default_rng(seed)
    indices = rng.permutation(total_rows)[:n]
    input_col = table.column("input_ids")
    easy_col = table.column("easiness_score")

    examples = []
    for i, row_idx in enumerate(indices.tolist()):
        ids = torch.tensor(input_col[row_idx].as_py(), dtype=torch.long)[:seq_len]
        if ids.numel() < seq_len:
            pad = torch.full(
                (seq_len - ids.numel(),),
                int(config.pad_token_id),
                dtype=torch.long,
            )
            ids = torch.cat([ids, pad], dim=0)

        masked, labels = deterministic_span_mask(
            ids,
            config,
            seed=seed + 100003 * i,
        )
        examples.append(
            {
                "input_ids": masked,
                "labels": labels,
                "attention_mask": (masked != int(config.pad_token_id)).long(),
                "easiness_score": torch.tensor(float(easy_col[row_idx].as_py()), dtype=torch.float32),
                "example_id": torch.tensor(i, dtype=torch.long),
            }
        )

    # Keep static batch shapes for XLA.
    usable = (len(examples) // batch_size) * batch_size
    examples = examples[:usable]
    if not examples:
        raise RuntimeError("Not enough examples for one complete batch")

    batches = []
    for start in range(0, len(examples), batch_size):
        chunk = examples[start : start + batch_size]
        batches.append({k: torch.stack([x[k] for x in chunk], dim=0) for k in chunk[0]})

    print(
        f"Prepared {len(examples)} examples -> {len(batches)} batches of {batch_size}, "
        f"seq_len={seq_len}"
    )
    return batches


def move_batch(batch: Dict[str, torch.Tensor], dev: DeviceContext) -> Dict[str, torch.Tensor]:
    return {k: v.to(dev.device) for k, v in batch.items()}


# -----------------------------------------------------------------------------
# CE helpers
# -----------------------------------------------------------------------------
def ce_sum_and_count(logits: torch.Tensor, labels: torch.Tensor, chunk_tokens: int = 128):
    total = logits.new_zeros((), dtype=torch.float32)
    count = labels.new_zeros((), dtype=torch.long)
    seq_len = logits.size(1)
    vocab = logits.size(-1)
    for start in range(0, seq_len, chunk_tokens):
        end = min(start + chunk_tokens, seq_len)
        lgt = logits[:, start:end, :].float().reshape(-1, vocab)
        lab = labels[:, start:end].reshape(-1)
        total = total + F.cross_entropy(lgt, lab, ignore_index=-100, reduction="sum")
        count = count + (lab != -100).sum()
    return total, count


def forward_ce(model, batch: Dict[str, torch.Tensor], dev: DeviceContext, checkpoint_step: int):
    with dev.autocast():
        logits, count_loss = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            current_step=checkpoint_step,
            easiness_score=batch["easiness_score"],
        )
        ce_sum, ce_count = ce_sum_and_count(logits, batch["labels"])
        ce = ce_sum / ce_count.clamp_min(1).to(ce_sum.dtype)
    return ce, ce_sum, ce_count, count_loss


def evaluate_ce(
    model,
    batches,
    dev: DeviceContext,
    checkpoint_step: int,
    max_batches: Optional[int] = None,
) -> float:
    total = 0.0
    count = 0
    use = batches if max_batches is None else batches[:max_batches]
    with torch.no_grad():
        for cpu_batch in use:
            batch = move_batch(cpu_batch, dev)
            _, ce_sum, ce_count, _ = forward_ce(model, batch, dev, checkpoint_step)
            dev.mark_step()
            total += float(ce_sum.detach().cpu())
            count += int(ce_count.detach().cpu())
    return total / max(1, count)


# -----------------------------------------------------------------------------
# Router override: only for CE controls
# -----------------------------------------------------------------------------
class RouterOverride:
    """Replace each 7c router output while preserving tensor shapes."""

    def __init__(self, model, mode: str, permanent_heads: int):
        self.model = model
        self.mode = mode
        self.permanent_heads = permanent_heads
        self.handles = []

    def _hook(self, layer_idx: int):
        def hook(module, inputs, output):
            b, h, _, _ = output.shape
            p = self.permanent_heads

            if self.mode == "dense":
                return torch.ones_like(output)

            if self.mode == "permanent_only":
                result = torch.zeros_like(output)
                result[:, :p] = 1.0
                return result

            if self.mode == "random_same_count":
                elastic = output[:, p:, 0, 0]
                k = (elastic > 0.5).sum(dim=-1, keepdim=True)
                noise = torch.rand_like(elastic.float())
                ranks = double_argsort_rank(noise, descending=True)
                rand_elastic = (ranks < k).to(output.dtype)
                permanent = torch.ones((b, p), device=output.device, dtype=output.dtype)
                return torch.cat([permanent, rand_elastic], dim=-1).view(b, h, 1, 1)

            raise ValueError(f"Unknown override mode: {self.mode}")

        return hook

    def __enter__(self):
        for i, block in enumerate(self.model.model.blocks):
            self.handles.append(block.mlt_vw_rtr.register_forward_hook(self._hook(i)))
        return self

    def __exit__(self, exc_type, exc, tb):
        for handle in self.handles:
            handle.remove()
        self.handles.clear()


def evaluate_override_mode(
    model,
    batches,
    dev: DeviceContext,
    checkpoint_step: int,
    mode: str,
    seed: int,
) -> float:
    # Re-seed so random-same-count is deterministic between runs.
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    with RouterOverride(model, mode, model.config.num_permanent_heads):
        return evaluate_ce(model, batches, dev, checkpoint_step)


# -----------------------------------------------------------------------------
# Router statistics
# -----------------------------------------------------------------------------
def collect_router_statistics(
    model,
    batches,
    dev: DeviceContext,
    checkpoint_step: int,
    output_dir: Path,
) -> dict:
    L = len(model.model.blocks)
    P = model.config.num_permanent_heads
    E = model.config.num_attention_heads - P

    logits_by_layer = [[] for _ in range(L)]
    sig_by_layer = [[] for _ in range(L)]
    mask_by_layer = [[] for _ in range(L)]
    actual_by_layer = [[] for _ in range(L)]
    target_by_layer = [[] for _ in range(L)]
    easiness_all = []

    with torch.no_grad():
        for cpu_batch in batches:
            batch = move_batch(cpu_batch, dev)
            _ = forward_ce(model, batch, dev, checkpoint_step)
            dev.mark_step()
            easiness_all.append(cpu_batch["easiness_score"].numpy())

            for li, block in enumerate(model.model.blocks):
                r = block.mlt_vw_rtr
                logits_by_layer[li].append(r.save_router_logits.detach().float().cpu().numpy())
                sig_by_layer[li].append(r.save_sigmoid_scores.detach().float().cpu().numpy())
                mask_by_layer[li].append(r.save_hard_mask.detach().float().cpu().numpy())
                actual_by_layer[li].append(r.save_total_head_count.detach().float().cpu().numpy())
                target_by_layer[li].append(r.save_target_total_head_count.detach().float().cpu().numpy())

    easiness = np.concatenate(easiness_all, axis=0)
    layer_rows = []
    head_rows = []
    arrays = {
        "logits": [],
        "sigmoid": [],
        "elastic_mask": [],
        "actual": [],
        "target": [],
    }

    for li, block in enumerate(model.model.blocks):
        logits = np.concatenate(logits_by_layer[li], axis=0)
        sig = np.concatenate(sig_by_layer[li], axis=0)
        mask = np.concatenate(mask_by_layer[li], axis=0)
        actual = np.concatenate(actual_by_layer[li], axis=0)
        target = np.concatenate(target_by_layer[li], axis=0)
        freq = mask.mean(axis=0)
        weight_norms = block.mlt_vw_rtr.q_up_proj.weight.detach().float().norm(dim=1).cpu().numpy()

        arrays["logits"].append(logits)
        arrays["sigmoid"].append(sig)
        arrays["elastic_mask"].append(mask)
        arrays["actual"].append(actual)
        arrays["target"].append(target)

        layer_rows.append(
            {
                "layer": li,
                "actual_mean": float(actual.mean()),
                "target_mean": float(target.mean()),
                "count_error_mean": float((actual - target).mean()),
                "count_mae": float(np.abs(actual - target).mean()),
                "easiness_actual_spearman": spearman_np(easiness, actual),
                "target_actual_spearman": spearman_np(target, actual),
                "sigmoid_mean": float(sig.mean()),
                "sigmoid_near_half_45_55": float(((sig > 0.45) & (sig < 0.55)).mean()),
                "sigmoid_saturation_lt05_gt95": float(((sig < 0.05) | (sig > 0.95)).mean()),
                "ste_derivative_mean": float((sig * (1.0 - sig)).mean()),
                "dynamic_elastic_fraction_05_95": float(((freq > 0.05) & (freq < 0.95)).mean()),
                "always_off_elastic_fraction": float((freq < 0.01).mean()),
                "always_on_elastic_fraction": float((freq > 0.99).mean()),
                "router_weight_norm_mean": float(weight_norms.mean()),
                "router_weight_norm_std": float(weight_norms.std()),
            }
        )

        for eh in range(E):
            head_rows.append(
                {
                    "layer": li,
                    "elastic_head": eh,
                    "absolute_head": eh + P,
                    "activation_frequency": float(freq[eh]),
                    "mean_logit": float(logits[:, eh].mean()),
                    "mean_sigmoid": float(sig[:, eh].mean()),
                    "router_weight_norm": float(weight_norms[eh]),
                }
            )

    with (output_dir / "router_layer_summary.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=layer_rows[0].keys())
        writer.writeheader()
        writer.writerows(layer_rows)

    with (output_dir / "router_head_summary.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=head_rows[0].keys())
        writer.writeheader()
        writer.writerows(head_rows)

    return {
        "easiness": easiness,
        "per_layer": {r["layer"]: r for r in layer_rows},
        "head_rows": head_rows,
        **arrays,
    }


# -----------------------------------------------------------------------------
# Capture attention inputs and recompute every head before routing
# -----------------------------------------------------------------------------
class AttentionInputCapture:
    def __init__(self, model, layers: Sequence[int]):
        self.model = model
        self.layers = set(layers)
        self.handles = []
        self.data: Dict[int, Tuple[torch.Tensor, torch.Tensor]] = {}

    def _hook(self, li: int):
        def hook(module, inputs):
            # HELM_7c attention signature:
            # forward(hidden_states, attention_mask, router_mask)
            self.data[li] = (inputs[0].detach(), inputs[1].detach())
        return hook

    def __enter__(self):
        for li in self.layers:
            self.handles.append(
                self.model.model.blocks[li].attn.register_forward_pre_hook(self._hook(li))
            )
        return self

    def __exit__(self, exc_type, exc, tb):
        for handle in self.handles:
            handle.remove()
        self.handles.clear()


def unmasked_attention_context(attn, hidden_states: torch.Tensor, attention_mask: torch.Tensor):
    """Recompute ALL heads exactly through attention, stopping before router masking."""
    qkv_proj = cast_linear(hidden_states, attn.qkv)
    b, s, _ = hidden_states.shape
    q, k, v = qkv_proj.split(attn.total_head_dim, dim=-1)

    q = q.view(b, s, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    k = k.view(b, s, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    v = v.view(b, s, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)

    q = justnorm(q)
    k = justnorm(k)
    q = attn.RoPE(q)
    k = attn.RoPE(k)

    sqk = attn.sqk * (attn.ngpt_sqk_init_value / attn.ngpt_sqk_init_scale)
    sqk = sqk.view(1, attn.num_attention_heads, 1, attn.d_head).to(q.dtype)
    q = sqk * q
    k = sqk * k

    context = F.scaled_dot_product_attention(
        q,
        k,
        v,
        attn_mask=attention_mask.to(q.dtype),
        scale=math.sqrt(attn.d_head),
    )

    if attn.config.use_exclusive_attention:
        vn = F.normalize(v, dim=-1)
        context = context - (context * vn).sum(dim=-1, keepdim=True) * vn

    return context


def full_hard_mask_for_layer(model, li: int, dtype: torch.dtype, device: torch.device):
    """Build [B,H,1,1] from the router's saved hard elastic mask."""
    block = model.model.blocks[li]
    elastic = block.mlt_vw_rtr.save_hard_mask.detach().to(device=device, dtype=dtype)
    b = elastic.size(0)
    p = model.config.num_permanent_heads
    if p > 0:
        permanent = torch.ones((b, p), device=device, dtype=dtype)
        full = torch.cat([permanent, elastic], dim=-1)
    else:
        full = elastic
    return full.view(b, model.config.num_attention_heads, 1, 1)


# -----------------------------------------------------------------------------
# Geometry/rank helpers
# -----------------------------------------------------------------------------
def group_indices(H: int, P: int) -> Dict[str, np.ndarray]:
    groups = {"all": np.arange(H, dtype=np.int64)}
    if P > 0:
        groups["permanent"] = np.arange(0, P, dtype=np.int64)
    if P < H:
        groups["elastic"] = np.arange(P, H, dtype=np.int64)
    return groups


def summarize_group_gram(gram: np.ndarray, idx: np.ndarray) -> dict:
    sub = gram[np.ix_(idx, idx)]
    cos = cosine_from_gram(sub)
    raw_er, raw_pr, raw_eig = effective_rank_from_gram(sub)
    dir_er, dir_pr, dir_eig = effective_rank_from_gram(cos)
    energy = np.clip(np.diag(sub), 0.0, None)
    frac = energy / (energy.sum() + 1e-12)
    stats = offdiag_stats(cos)
    return {
        "n_heads": int(len(idx)),
        "raw_entropy_rank": raw_er,
        "raw_participation_rank": raw_pr,
        "directional_entropy_rank": dir_er,
        "directional_participation_rank": dir_pr,
        "mean_abs_cosine": stats["mean_abs"],
        "median_abs_cosine": stats["median_abs"],
        "max_abs_cosine": stats["max_abs"],
        "frac_abs_cos_gt_05": stats["frac_abs_gt_0.5"],
        "frac_abs_cos_gt_08": stats["frac_abs_gt_0.8"],
        "top1_energy_fraction": float(frac.max()) if frac.size else 0.0,
        "top4_energy_fraction": float(np.sort(frac)[-min(4, len(frac)):].sum()) if frac.size else 0.0,
        "energy_gini": gini_nonnegative(energy),
        "components_90pct": components_for_fraction(raw_eig, 0.90),
        "components_95pct": components_for_fraction(raw_eig, 0.95),
        "raw_eigenvalues": raw_eig,
        "directional_eigenvalues": dir_eig,
        "cosine_matrix": cos,
    }


def head_bank_audit(
    model,
    batches,
    dev: DeviceContext,
    checkpoint_step: int,
    max_batches: int,
    sample_tokens: int,
    output_dir: Path,
    plot_layers: Sequence[int],
):
    """
    Measure potential and executed geometry for all/permanent/elastic groups.

    potential: every head recomputed before routing
    executed : same head output multiplied by actual hard 7c mask
    """
    layers = list(range(len(model.model.blocks)))
    H = model.config.num_attention_heads
    P = model.config.num_permanent_heads
    E = H - P

    # geometry -> representation -> layer -> Gram
    grams = {
        geometry: {
            rep: {li: np.zeros((H, H), dtype=np.float64) for li in layers}
            for rep in ("context", "residual")
        }
        for geometry in ("potential", "executed")
    }
    counts = {
        geometry: {
            rep: {li: 0 for li in layers}
            for rep in ("context", "residual")
        }
        for geometry in ("potential", "executed")
    }

    activation_sum = {li: np.zeros(H, dtype=np.float64) for li in layers}
    activation_n = {li: 0 for li in layers}

    with torch.no_grad():
        for cpu_batch in batches[:max_batches]:
            batch = move_batch(cpu_batch, dev)

            # One real routed pass gives us the hidden state entering each attention
            # layer and the actual hard mask selected for this exact input.
            with AttentionInputCapture(model, layers) as cap:
                with dev.autocast():
                    _ = model(
                        input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"],
                        current_step=checkpoint_step,
                        easiness_score=batch["easiness_score"],
                    )
                dev.mark_step()

            for li in layers:
                hidden, attn_mask = cap.data[li]
                attn = model.model.blocks[li].attn

                with dev.autocast():
                    context = unmasked_attention_context(attn, hidden, attn_mask)  # [B,H,S,d]
                    hard = full_hard_mask_for_layer(model, li, context.dtype, context.device)

                    seq = context.size(2)
                    t = min(sample_tokens, seq)
                    positions = torch.linspace(0, seq - 1, steps=t, device=context.device).long()
                    c_potential = context.index_select(2, positions)  # [B,H,T,d]
                    c_executed = c_potential * hard

                    # W_O block per head: [D,H,d]
                    W = attn.output.weight.to(c_potential.dtype).view(
                        attn.hidden_size, H, attn.d_head
                    )
                    y_potential = torch.einsum("bhtd,ohd->bhto", c_potential, W)
                    y_executed = torch.einsum("bhtd,ohd->bhto", c_executed, W)

                    tensors = {
                        ("potential", "context"): c_potential,
                        ("potential", "residual"): y_potential,
                        ("executed", "context"): c_executed,
                        ("executed", "residual"): y_executed,
                    }

                    local_grams = {}
                    local_counts = {}
                    for key, tensor in tensors.items():
                        flat = tensor.permute(1, 0, 2, 3).contiguous().view(H, -1).float()
                        local_grams[key] = flat @ flat.T
                        local_counts[key] = int(flat.shape[1])

                dev.mark_step()

                for (geometry, rep), gram_t in local_grams.items():
                    grams[geometry][rep][li] += (gram_t.detach().to(torch.float32).cpu().numpy().astype(np.float64))
                    counts[geometry][rep][li] += local_counts[(geometry, rep)]

                hard_np = hard[:, :, 0, 0].detach().float().cpu().numpy()
                activation_sum[li] += hard_np.sum(axis=0)
                activation_n[li] += hard_np.shape[0]

                del context, c_potential, c_executed, y_potential, y_executed, local_grams

    groups = group_indices(H, P)
    rank_rows = []
    per_layer = {}
    head_rows = []

    # Router arrays collected over the full router-stat batches.  Head-bank audit may
    # use fewer batches, so activation_frequency here is specific to functional subset.
    for li in layers:
        attn = model.model.blocks[li].attn
        per_layer[li] = {}

        # Head-level potential/executed magnitude.
        p_c_energy = np.clip(np.diag(grams["potential"]["context"][li]), 0.0, None)
        p_r_energy = np.clip(np.diag(grams["potential"]["residual"][li]), 0.0, None)
        e_r_energy = np.clip(np.diag(grams["executed"]["residual"][li]), 0.0, None)

        context_rms = np.sqrt(
            p_c_energy / max(1, counts["potential"]["context"][li])
        )
        residual_rms = np.sqrt(
            p_r_energy / max(1, counts["potential"]["residual"][li])
        )
        executed_residual_rms = np.sqrt(
            e_r_energy / max(1, counts["executed"]["residual"][li])
        )
        activation_frequency = activation_sum[li] / max(1, activation_n[li])

        residual_fraction_all = p_r_energy / (p_r_energy.sum() + 1e-12)

        w = attn.output.weight.detach().float().cpu().numpy().reshape(
            attn.hidden_size, H, attn.d_head
        )
        w_block_frob = np.sqrt(np.square(w).sum(axis=(0, 2)))

        qkv = attn.qkv.weight.detach().float().cpu().numpy().reshape(
            3, H, attn.d_head, attn.hidden_size
        )
        qkv_block_frob = np.sqrt(np.square(qkv).sum(axis=(0, 2, 3)))

        # Router stats per elastic head from current model state / functional subset.
        router = model.model.blocks[li].mlt_vw_rtr
        elastic_sig = router.save_sigmoid_scores.detach().float().cpu().numpy()
        elastic_logits = router.save_router_logits.detach().float().cpu().numpy()
        # Those saved arrays are only the LAST functional batch; keep head-level means
        # here as a local reference, while activation_frequency is aggregated.
        mean_sig_last = elastic_sig.mean(axis=0)
        mean_logit_last = elastic_logits.mean(axis=0)

        for h in range(H):
            is_perm = h < P
            eh = h - P
            head_rows.append(
                {
                    "layer": li,
                    "head": h,
                    "group": "permanent" if is_perm else "elastic",
                    "activation_frequency_functional_subset": float(activation_frequency[h]),
                    "context_rms_potential": float(context_rms[h]),
                    "residual_rms_potential": float(residual_rms[h]),
                    "residual_rms_executed": float(executed_residual_rms[h]),
                    "residual_energy_fraction_all_heads_potential": float(residual_fraction_all[h]),
                    "output_block_frobenius": float(w_block_frob[h]),
                    "qkv_block_frobenius": float(qkv_block_frob[h]),
                    "last_batch_mean_router_sigmoid": 1.0 if is_perm else float(mean_sig_last[eh]),
                    "last_batch_mean_router_logit": float("nan") if is_perm else float(mean_logit_last[eh]),
                }
            )

        # Group rank summaries.
        for geometry in ("potential", "executed"):
            for rep in ("context", "residual"):
                full_gram = grams[geometry][rep][li]
                for group_name, idx in groups.items():
                    s = summarize_group_gram(full_gram, idx)
                    row = {
                        "layer": li,
                        "geometry": geometry,
                        "representation": rep,
                        "group": group_name,
                        "n_heads": s["n_heads"],
                        "raw_entropy_rank": s["raw_entropy_rank"],
                        "raw_participation_rank": s["raw_participation_rank"],
                        "directional_entropy_rank": s["directional_entropy_rank"],
                        "directional_participation_rank": s["directional_participation_rank"],
                        "mean_abs_cosine": s["mean_abs_cosine"],
                        "median_abs_cosine": s["median_abs_cosine"],
                        "max_abs_cosine": s["max_abs_cosine"],
                        "top1_energy_fraction": s["top1_energy_fraction"],
                        "top4_energy_fraction": s["top4_energy_fraction"],
                        "energy_gini": s["energy_gini"],
                        "components_90pct": s["components_90pct"],
                        "components_95pct": s["components_95pct"],
                    }
                    rank_rows.append(row)
                    per_layer[li][(geometry, rep, group_name)] = {**row, **s}

        # Correlations central to the original starvation/strength question.
        elastic_idx = np.arange(P, H)
        if E > 0:
            per_layer[li]["elastic_correlations"] = {
                "activation_vs_potential_residual_rms_spearman": spearman_np(
                    activation_frequency[elastic_idx], residual_rms[elastic_idx]
                ),
                "activation_vs_executed_residual_rms_spearman": spearman_np(
                    activation_frequency[elastic_idx], executed_residual_rms[elastic_idx]
                ),
                "activation_vs_wo_norm_spearman": spearman_np(
                    activation_frequency[elastic_idx], w_block_frob[elastic_idx]
                ),
            }

        # Selected-layer plots only, to keep output compact.
        if li in plot_layers:
            p_res_all = per_layer[li][("potential", "residual", "all")]
            p_res_elastic = per_layer[li][("potential", "residual", "elastic")]
            e_res_all = per_layer[li][("executed", "residual", "all")]

            save_heatmap(
                output_dir / f"layer_{li:02d}_potential_residual_cosine_all.png",
                p_res_all["cosine_matrix"],
                f"Layer {li}: potential residual cosine (all heads)",
                -1.0,
                1.0,
            )
            save_heatmap(
                output_dir / f"layer_{li:02d}_potential_residual_cosine_elastic.png",
                p_res_elastic["cosine_matrix"],
                f"Layer {li}: potential residual cosine (elastic heads)",
                -1.0,
                1.0,
            )
            save_heatmap(
                output_dir / f"layer_{li:02d}_executed_residual_cosine_all.png",
                e_res_all["cosine_matrix"],
                f"Layer {li}: executed residual cosine (all heads)",
                -1.0,
                1.0,
            )
            save_bar(
                output_dir / f"layer_{li:02d}_potential_residual_rms.png",
                residual_rms,
                f"Layer {li}: potential residual RMS",
                "Residual RMS",
            )
            save_bar(
                output_dir / f"layer_{li:02d}_activation_frequency.png",
                activation_frequency,
                f"Layer {li}: hard activation frequency (permanent + elastic)",
                "Activation frequency",
            )

    with (output_dir / "rank_summary_all_layers.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rank_rows[0].keys())
        writer.writeheader()
        writer.writerows(rank_rows)

    with (output_dir / "head_metrics_all_layers.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=head_rows[0].keys())
        writer.writeheader()
        writer.writerows(head_rows)

    # A few layer-depth plots for the exact metrics we now care about.
    residual_potential = [
        r for r in rank_rows
        if r["geometry"] == "potential" and r["representation"] == "residual"
    ]
    context_potential = [
        r for r in rank_rows
        if r["geometry"] == "potential" and r["representation"] == "context"
    ]
    save_rank_depth_plot(
        output_dir / "potential_residual_raw_rank_by_depth.png",
        residual_potential,
        "raw_entropy_rank",
        "HELM_7c potential residual RAW effective rank",
    )
    save_rank_depth_plot(
        output_dir / "potential_residual_directional_rank_by_depth.png",
        residual_potential,
        "directional_entropy_rank",
        "HELM_7c potential residual DIRECTIONAL effective rank",
    )
    save_rank_depth_plot(
        output_dir / "potential_context_directional_rank_by_depth.png",
        context_potential,
        "directional_entropy_rank",
        "HELM_7c potential pre-W_O DIRECTIONAL effective rank",
    )

    return {
        "per_layer": per_layer,
        "rank_rows": rank_rows,
        "head_rows": head_rows,
    }


# -----------------------------------------------------------------------------
# Optional exact single-head ablation under NORMAL 7c routing
# -----------------------------------------------------------------------------
@contextmanager
def temporarily_zero_output_heads(model, dropped_by_layer: Dict[int, Sequence[int]]):
    backups = {}
    try:
        with torch.no_grad():
            for li, heads in dropped_by_layer.items():
                attn = model.model.blocks[li].attn
                cols = []
                for h in heads:
                    start = int(h) * attn.d_head
                    cols.extend(range(start, start + attn.d_head))
                if not cols:
                    continue
                idx = torch.tensor(cols, device=attn.output.weight.device, dtype=torch.long)
                backups[li] = (idx, attn.output.weight.index_select(1, idx).clone())
                attn.output.weight[:, idx] = 0
        yield
    finally:
        with torch.no_grad():
            for li, (idx, backup) in backups.items():
                model.model.blocks[li].attn.output.weight[:, idx] = backup


def exact_single_head_ablation(
    model,
    batches,
    dev: DeviceContext,
    checkpoint_step: int,
    layers: Sequence[int],
    max_batches: int,
    geometry: dict,
    output_dir: Path,
):
    subset = batches[:max_batches]
    baseline_ce = evaluate_ce(model, subset, dev, checkpoint_step)
    H = model.config.num_attention_heads
    P = model.config.num_permanent_heads

    # Look up potential residual RMS by layer/head.
    rms_lookup = {
        (int(r["layer"]), int(r["head"])): float(r["residual_rms_potential"])
        for r in geometry["head_rows"]
    }
    act_lookup = {
        (int(r["layer"]), int(r["head"])): float(r["activation_frequency_functional_subset"])
        for r in geometry["head_rows"]
    }

    rows = []
    summaries = {}
    for li in layers:
        for h in range(H):
            with temporarily_zero_output_heads(model, {li: [h]}):
                ce = evaluate_ce(model, subset, dev, checkpoint_step)
            delta = ce - baseline_ce
            rows.append(
                {
                    "layer": li,
                    "head": h,
                    "group": "permanent" if h < P else "elastic",
                    "baseline_routed_ce": baseline_ce,
                    "ablated_routed_ce": ce,
                    "delta_ce": delta,
                    "activation_frequency_functional_subset": act_lookup[(li, h)],
                    "potential_residual_rms": rms_lookup[(li, h)],
                }
            )
            print(f"Exact routed ablation L{li:02d} H{h:02d}: ΔCE={delta:+.6f}")

        rr = [r for r in rows if r["layer"] == li]
        for group in ("all", "permanent", "elastic"):
            gg = rr if group == "all" else [r for r in rr if r["group"] == group]
            if not gg:
                continue
            d = np.array([r["delta_ce"] for r in gg], dtype=np.float64)
            rms = np.array([r["potential_residual_rms"] for r in gg], dtype=np.float64)
            act = np.array([r["activation_frequency_functional_subset"] for r in gg], dtype=np.float64)
            summaries[(li, group)] = {
                "mean_delta_ce": float(d.mean()),
                "median_delta_ce": float(np.median(d)),
                "max_delta_ce": float(d.max()),
                "min_delta_ce": float(d.min()),
                "fraction_delta_gt_0": float((d > 0).mean()),
                "rms_vs_delta_ce_spearman": spearman_np(rms, d),
                "activation_vs_delta_ce_spearman": spearman_np(act, d),
            }

    with (output_dir / "exact_single_head_ablation.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)

    return {
        "baseline_routed_ce_subset": baseline_ce,
        "rows": rows,
        "summaries": summaries,
    }


# -----------------------------------------------------------------------------
# Reporting
# -----------------------------------------------------------------------------
def get_rank(geometry: dict, li: int, geometry_name: str, rep: str, group: str) -> dict:
    return geometry["per_layer"][li][(geometry_name, rep, group)]


def write_summary(
    output_dir: Path,
    args,
    checkpoint_name: str,
    config,
    router_stats: dict,
    ce_modes: dict,
    geometry: dict,
    exact,
):
    H = config.num_attention_heads
    P = config.num_permanent_heads
    E = H - P
    L = len(geometry["per_layer"])

    lines = []
    lines.append("# HELM_7c head-bank assumption audit\n\n")
    lines.append(f"Checkpoint: **{checkpoint_name}**  \n")
    lines.append(f"Heads: **{H} total = {P} permanent + {E} elastic**  \n")
    lines.append(
        "This audit separates **raw rank** (direction + magnitude) from "
        "**directional rank** (direction only), and separates the full "
        "**potential** head bank from the actually **executed** routed bank.\n\n"
    )

    lines.append("## 1. CE controls\n\n")
    lines.append("|Mode|CE|Δ vs routed|\n|---|---:|---:|\n")
    routed = ce_modes["routed"]
    for mode in ("routed", "dense", "random_same_count", "permanent_only"):
        ce = ce_modes[mode]
        lines.append(f"|{mode}|{ce:.6f}|{ce-routed:+.6f}|\n")
    lines.append(
        "\nInterpretation: **random-same-count** tests whether identity matters at fixed compute; "
        "**dense** asks whether this routed-trained checkpoint benefits from turning every head on; "
        "**permanent-only** tests whether the elastic bank is genuinely needed.\n\n"
    )

    lines.append("## 2. Router sanity\n\n")
    lines.append("|Layer|Actual K|Target K|MAE|rho(target,actual)|sigmoid mean|near .5|\n")
    lines.append("|---:|---:|---:|---:|---:|---:|---:|\n")
    for li in range(L):
        r = router_stats["per_layer"][li]
        lines.append(
            f"|{li}|{r['actual_mean']:.2f}|{r['target_mean']:.2f}|{r['count_mae']:.2f}|"
            f"{r['target_actual_spearman']:.3f}|{r['sigmoid_mean']:.3f}|"
            f"{100*r['sigmoid_near_half_45_55']:.1f}%|\n"
        )

    lines.append("\n## 3. The key comparison: POTENTIAL residual rank\n\n")
    lines.append(
        "Potential means **all heads are recomputed before masking**. This is the closest "
        "apples-to-apples comparison to the dense-16 head-bank audit.\n\n"
    )
    lines.append(
        "|Layer|All raw|All dir|Permanent raw|Permanent dir|Elastic raw|Elastic dir|Elastic mean |cos||\n"
    )
    lines.append("|---:|---:|---:|---:|---:|---:|---:|---:|\n")
    for li in range(L):
        a = get_rank(geometry, li, "potential", "residual", "all")
        p = get_rank(geometry, li, "potential", "residual", "permanent")
        e = get_rank(geometry, li, "potential", "residual", "elastic")
        lines.append(
            f"|{li}|{a['raw_entropy_rank']:.2f}/{H}|{a['directional_entropy_rank']:.2f}/{H}|"
            f"{p['raw_entropy_rank']:.2f}/{P}|{p['directional_entropy_rank']:.2f}/{P}|"
            f"{e['raw_entropy_rank']:.2f}/{E}|{e['directional_entropy_rank']:.2f}/{E}|"
            f"{e['mean_abs_cosine']:.3f}|\n"
        )

    lines.append("\n### How to interpret this table\n")
    lines.append(
        "- **low elastic raw + high elastic directional rank** → the elastic heads are mostly "
        "different directions but have unequal strength. That is much closer to what the dense-16 "
        "baseline does and is **not** evidence for duplicate-head collapse.\n"
    )
    lines.append(
        "- **low elastic raw + low elastic directional rank** → real directional redundancy exists "
        "inside the elastic candidate bank. That would be a stronger reason to consider a diversity objective.\n"
    )
    lines.append(
        "- **context directional rank high but residual directional rank low** → attention contexts "
        "are diverse, but W_O aligns/compresses how they write back to the residual stream.\n"
    )

    lines.append("\n## 4. Pre-W_O vs post-W_O directional rank (POTENTIAL elastic bank)\n\n")
    lines.append("|Layer|Context dir rank|Residual dir rank|Context mean |cos||Residual mean |cos||\n")
    lines.append("|---:|---:|---:|---:|---:|\n")
    for li in range(L):
        c = get_rank(geometry, li, "potential", "context", "elastic")
        r = get_rank(geometry, li, "potential", "residual", "elastic")
        lines.append(
            f"|{li}|{c['directional_entropy_rank']:.2f}/{E}|{r['directional_entropy_rank']:.2f}/{E}|"
            f"{c['mean_abs_cosine']:.3f}|{r['mean_abs_cosine']:.3f}|\n"
        )

    lines.append("\n## 5. Potential vs executed elastic bank\n\n")
    lines.append(
        "Potential asks **are the candidates diverse?** Executed asks **what does hard routing "
        "actually expose across these examples?** A large potential→executed drop can be a routing "
        "selection effect rather than a representational collapse.\n\n"
    )
    lines.append("|Layer|Potential raw|Potential dir|Executed raw|Executed dir|\n")
    lines.append("|---:|---:|---:|---:|---:|\n")
    for li in range(L):
        p = get_rank(geometry, li, "potential", "residual", "elastic")
        x = get_rank(geometry, li, "executed", "residual", "elastic")
        lines.append(
            f"|{li}|{p['raw_entropy_rank']:.2f}/{E}|{p['directional_entropy_rank']:.2f}/{E}|"
            f"{x['raw_entropy_rank']:.2f}/{E}|{x['directional_entropy_rank']:.2f}/{E}|\n"
        )

    lines.append("\n## 6. Does usage predict strength?\n\n")
    for li in range(L):
        c = geometry["per_layer"][li].get("elastic_correlations", {})
        lines.append(
            f"- L{li:02d}: rho(activation, **potential** residual RMS)="
            f"{c.get('activation_vs_potential_residual_rms_spearman', float('nan')):.3f}; "
            f"rho(activation, **executed** residual RMS)="
            f"{c.get('activation_vs_executed_residual_rms_spearman', float('nan')):.3f}.\n"
        )
    lines.append(
        "This correlation is descriptive, not causal. 7d showed why increasing gradients to OFF "
        "heads without forward exposure is not a valid way to infer causality.\n"
    )

    lines.append("\n## 7. Exact routed single-head ablation\n\n")
    if exact is None:
        lines.append("Skipped. Re-run with `--exact-ablation` if needed.\n")
    else:
        lines.append(
            f"Baseline routed CE on ablation subset: **{exact['baseline_routed_ce_subset']:.6f}**\n\n"
        )
        for (li, group), s in sorted(exact["summaries"].items(), key=lambda kv: (kv[0][0], kv[0][1])):
            lines.append(
                f"- L{li:02d} {group}: mean ΔCE={s['mean_delta_ce']:+.6f}, "
                f"max={s['max_delta_ce']:+.6f}, rho(RMS,ΔCE)={s['rms_vs_delta_ce_spearman']:.3f}, "
                f"rho(activation,ΔCE)={s['activation_vs_delta_ce_spearman']:.3f}.\n"
            )

    lines.append("\n## 8. Assumptions this script is explicitly checking\n\n")
    lines.append("1. **'7c has low rank, therefore its heads are redundant.'** Raw rank alone cannot establish that.\n")
    lines.append("2. **'Unequal head energy is a pathology.'** Dense-16 showed substantial raw-rank loss despite near-maximal directional rank.\n")
    lines.append("3. **'The router causes any observed rank loss.'** Potential vs executed rank separates candidate-bank geometry from selection geometry.\n")
    lines.append("4. **'Attention itself creates the redundancy.'** Context vs residual rank tests whether W_O is the stronger source of alignment.\n")
    lines.append("5. **'Frequently selected heads are inherently better heads.'** Usage-vs-strength and optional exact ablation test association without assuming causality.\n")
    lines.append("6. **'All 32 would be better if turned on.'** Routed-vs-forced-dense CE tests that checkpoint-level claim directly.\n")

    with (output_dir / "summary.md").open("w") as f:
        f.writelines(lines)


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser(
        description="HELM_7c raw-vs-directional head-rank assumption audit"
    )
    parser.add_argument("--model-repo", default=MODEL_REPO)
    parser.add_argument("--checkpoint", default=CHECKPOINT_FILE)
    parser.add_argument("--data-repo", default=DATA_REPO)
    parser.add_argument("--validation-file", default=VALIDATION_FILE)
    parser.add_argument("--device", default="auto", choices=["auto", "xla", "cuda", "cpu"])
    parser.add_argument("--num-examples", type=int, default=16)
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--seq-len", type=int, default=1024)
    parser.add_argument("--seed", type=int, default=1216)
    parser.add_argument("--functional-batches", type=int, default=4)
    parser.add_argument("--functional-sample-tokens", type=int, default=16)
    parser.add_argument("--plot-layers", default="0,5,11")
    parser.add_argument("--exact-ablation", action="store_true")
    parser.add_argument("--ablation-layers", default="0,5,11")
    parser.add_argument("--ablation-batches", type=int, default=1)
    parser.add_argument("--output-dir", default="helm7c_rank_audit")
    parser.add_argument("--cache-dir", default="./helm7c_rank_audit_cache")
    args = parser.parse_args()

    seed_everything(args.seed)
    dev = resolve_device(args.device)
    token = get_hf_token()

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path, training_state_path, validation_path = download_assets(
        Path(args.cache_dir),
        token,
        args.model_repo,
        args.checkpoint,
        args.data_repo,
        args.validation_file,
    )
    checkpoint_step = checkpoint_step_from_name(args.checkpoint)
    breakpoints = load_breakpoints(training_state_path)
    model, config = load_model(ckpt_path, breakpoints, dev)

    H = config.num_attention_heads
    P = config.num_permanent_heads
    E = H - P
    print(f"Architecture: H={H}, permanent={P}, elastic={E}, d_head={model.model.blocks[0].attn.d_head}")

    batches = prepare_batches(
        validation_path,
        config,
        args.num_examples,
        args.batch_size,
        args.seq_len,
        args.seed,
    )

    print("\n[1/4] Router statistics + normal routed CE...")
    router_stats = collect_router_statistics(
        model,
        batches,
        dev,
        checkpoint_step,
        output_dir,
    )
    routed_ce = evaluate_ce(model, batches, dev, checkpoint_step)
    print(f"Routed CE = {routed_ce:.6f}")

    print("\n[2/4] CE controls...")
    dense_ce = evaluate_override_mode(model, batches, dev, checkpoint_step, "dense", args.seed + 1)
    random_ce = evaluate_override_mode(
        model, batches, dev, checkpoint_step, "random_same_count", args.seed + 2
    )
    permanent_ce = evaluate_override_mode(
        model, batches, dev, checkpoint_step, "permanent_only", args.seed + 3
    )
    ce_modes = {
        "routed": routed_ce,
        "dense": dense_ce,
        "random_same_count": random_ce,
        "permanent_only": permanent_ce,
    }
    for k, v in ce_modes.items():
        print(f"  {k:18s}: {v:.6f}")

    print("\n[3/4] All-layer potential vs executed raw/directional rank...")
    plot_layers = parse_layers(args.plot_layers, len(model.model.blocks))
    geometry = head_bank_audit(
        model,
        batches,
        dev,
        checkpoint_step,
        max_batches=min(args.functional_batches, len(batches)),
        sample_tokens=args.functional_sample_tokens,
        output_dir=output_dir,
        plot_layers=plot_layers,
    )

    # Print the elastic result first because that is the question that motivated this rewrite.
    print("\nElastic POTENTIAL residual rank:")
    for li in range(len(model.model.blocks)):
        r = get_rank(geometry, li, "potential", "residual", "elastic")
        print(
            f"  L{li:02d}: raw={r['raw_entropy_rank']:.2f}/{E}, "
            f"directional={r['directional_entropy_rank']:.2f}/{E}, "
            f"mean|cos|={r['mean_abs_cosine']:.3f}, "
            f"top1 energy={100*r['top1_energy_fraction']:.1f}%"
        )

    exact = None
    if args.exact_ablation:
        print("\n[4/4] Exact single-head ablation under NORMAL 7c routing...")
        ablation_layers = parse_layers(args.ablation_layers, len(model.model.blocks))
        exact = exact_single_head_ablation(
            model,
            batches,
            dev,
            checkpoint_step,
            ablation_layers,
            min(args.ablation_batches, len(batches)),
            geometry,
            output_dir,
        )
    else:
        print("\n[4/4] Exact ablation skipped (use --exact-ablation if needed).")

    payload = {
        "model_repo": args.model_repo,
        "checkpoint": args.checkpoint,
        "checkpoint_step": checkpoint_step,
        "architecture": {
            "hidden_size": config.hidden_size,
            "num_attention_heads": H,
            "num_permanent_heads": P,
            "num_elastic_heads": E,
            "d_head": model.model.blocks[0].attn.d_head,
            "total_head_dim": model.model.blocks[0].attn.total_head_dim,
        },
        "ce_modes": ce_modes,
        "router_stats": router_stats,
        "geometry": geometry,
        "exact_ablation": exact,
    }

    with (output_dir / "summary.json").open("w") as f:
        json.dump(json_safe(payload), f, indent=2)

    write_summary(
        output_dir,
        args,
        args.checkpoint,
        config,
        router_stats,
        ce_modes,
        geometry,
        exact,
    )

    # Archive OUTSIDE the directory being archived (avoids the old recursive ZIP bug).
    archive_base = output_dir.parent / f"{output_dir.name}_results"
    archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=output_dir)

    print("\nDone.")
    print(f"Summary: {output_dir / 'summary.md'}")
    print(f"JSON:    {output_dir / 'summary.json'}")
    print(f"ZIP:     {archive_path}")


if __name__ == "__main__":
    main()

In [ ]:
!python analyze_helm7c_heads.py \
    --device xla \
    --batch-size 2 \
    --num-examples 8 \
    --utility-batches 4 \
    --functional-batches 4 \
    --exact-ablation \
    --exact-ablation-batches 2